Anime fix

In [1]:
import pandas as pd
from pathlib import Path

FOLDER = Path.cwd().parent / "data"
file_path = FOLDER / "anime.csv"

df = pd.read_csv(file_path)
df.head()

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,...,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,...,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,...,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,...,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,...,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [2]:
df.columns

Index(['MAL_ID', 'Name', 'Score', 'Genres', 'English name', 'Japanese name',
       'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors',
       'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity',
       'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped',
       'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6',
       'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1'],
      dtype='object')

In [3]:
df.drop(['Name', 'Score', 'Genres', 'Type', 'English name', 'Japanese name', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors',
       'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity',
       'Members','On-Hold', 'Dropped',
       'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6',
       'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1'], axis=1, inplace=True )

df.head()

,MAL_ID,Favorites,Watching,Completed
0,1,61971,105808,718161
1,5,1174,4143,208333
2,6,12944,29113,343492
3,7,587,4300,46165
4,8,18,642,7314


In [4]:
df.columns

Index(['MAL_ID', 'Favorites', 'Watching', 'Completed'], dtype='object')

In [5]:
df.to_csv(FOLDER / "add_cols.csv", index=False)

In [6]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(Path.cwd().parent / ".env")
engine = create_engine(os.getenv("DATABASE_URL"))

with engine.begin() as conn:
    db_ids = {r[0] for r in conn.execute(text("SELECT id FROM anime")).fetchall()}

print(f"db anime: {len(db_ids)}, csv rows before: {len(df)}")
df = df[df["MAL_ID"].isin(db_ids)].reset_index(drop=True)
print(f"csv rows after: {len(df)}")
df.to_csv(FOLDER / "add_cols.csv", index=False)

db anime: 16206, csv rows before: 17562
csv rows after: 16206


In [7]:
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE anime ADD COLUMN IF NOT EXISTS favorites INTEGER"))
    conn.execute(text("ALTER TABLE anime ADD COLUMN IF NOT EXISTS watching INTEGER"))
    conn.execute(text("ALTER TABLE anime ADD COLUMN IF NOT EXISTS completed INTEGER"))

    rows = [
        {
            "id": int(r.MAL_ID),
            "favorites": int(r.Favorites),
            "watching": int(r.Watching),
            "completed": int(r.Completed),
        }
        for r in df.itertuples(index=False)
    ]
    conn.execute(
        text(
            "UPDATE anime SET favorites = :favorites, watching = :watching, "
            "completed = :completed WHERE id = :id"
        ),
        rows,
    )
print(f"updated {len(rows)} anime rows")

updated 16206 anime rows
